# Quick SLM — 11 · Talk to the agent checkpoint

An interactive session with the fine-tuned model (`sft_final`). It builds the
exact ChatML envelope the model was trained on — system prompt, the tool
schemas, optional `<state>` and `<memory>` blocks, then your message — and shows
the model's `<think>` reasoning and the tool call it emits.

Two things this notebook gets right that a naive loop would not, both from the
evaluation in the paper:

- **Special tokens are preserved on decode.** `<think>`, `<response>` and the rest
  are registered tokens; decoding with `skip_special_tokens=True` would delete
  the whole envelope.
- **Generation is truncated at the first `</response>`.** The model does not stop
  after its call — it invents a tool result and keeps going — so only the first
  complete turn is kept.

The model is a 103M tool-caller, not a chatbot. It answers by choosing a tool and
filling its arguments; for anything outside its tools it should call `answer`.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install dependencies

In [ ]:
!pip -q install --upgrade transformers accelerate safetensors tokenizers

## 3. Load the checkpoint

In [ ]:
import sys, json
from pathlib import Path

REPO_DIR = Path('/content/drive/MyDrive/quick-slm/code')
framework_dir = REPO_DIR / 'framework'
assert framework_dir.is_dir(), f'{framework_dir} not found'
if str(framework_dir) not in sys.path:
    sys.path.insert(0, str(framework_dir))

from v1.quick_slm_trainer.support import require_framework
require_framework('v1', REPO_DIR)
from v1.quick_slm_trainer.paths import Layout

DRIVE_ROOT = Path('/content/drive/MyDrive/quick-slm')
layout = Layout(drive_root=DRIVE_ROOT)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

CKPT = layout.sft_final_dir          # the agent checkpoint; set to a step dir to try another
assert CKPT.is_dir(), f'{CKPT} not found -- run 05_sft_train first'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.bfloat16 if device == 'cuda' else torch.float32
tok = AutoTokenizer.from_pretrained(str(layout.tokenizer_dir))
model = AutoModelForCausalLM.from_pretrained(str(CKPT), dtype=dtype).to(device).eval()
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
print(f'loaded {CKPT.name}  ({sum(p.numel() for p in model.parameters())/1e6:.1f}M params) on {device}')

## 4. Choose the tools and the world

The model only knows the tools it was trained on. Two domains are available:

- **`world`** — general assistant tools: weather, capital cities, web search, time,
  currency and unit conversion, stock price, a calculator, indoor activities,
  population.
- **`factory`** — the factory-simulation tools: inspect, build, set recipe, pause
  and resume, buy and sell, power, research, and so on. Use this with a `<state>`
  block to test grounded calls.

`answer` is always available. Optionally set a `<state>` (the live world the
server would inject) and a `<memory>` (recent context). Leave them `None` for a
plain request.

In [ ]:
from v1.quick_slm_trainer.sft.tools import WORLD_TOOLS, FACTORY_TOOLS, ANSWER, DOMAIN_TOOLS
from v1.quick_slm_trainer.template import DEFAULT_SYSTEM

DOMAIN = 'world'                     # 'world' or 'factory'
TOOLS  = [ANSWER] + DOMAIN_TOOLS[DOMAIN]
SYSTEM = DEFAULT_SYSTEM

# Optional context blocks. Examples:
#   STATE  = {'buildings': {'b_004': {'type': 'smelter', 'status': 'paused'}}}
#   MEMORY = 'recent: the user asked about b_004 last turn'
STATE  = None
MEMORY = None

print(f'domain {DOMAIN!r}: {len(TOOLS)} tools available')
print('  ' + ', '.join(t['name'] for t in TOOLS))

## 5. The chat function

`chat("your message")` builds the prompt, generates, and prints the reasoning and
the call. Set `temperature` above 0 to sample; the default is greedy, which is
what the paper's evaluation uses.

In [ ]:
from v1.quick_slm_trainer.template import (Example, UserTurn, AssistantTurn,
                                        render_prompt, parse_think, parse_calls)

MAX_NEW = 192

def _build_prompt(message):
    # A throwaway assistant turn satisfies Example construction; render_prompt
    # stops before it and returns the priming up to '<|im_start|>assistant'.
    ex = Example(
        tools=TOOLS,
        turns=[UserTurn(message),
               AssistantTurn(think='.', calls=[{'name': 'answer', 'arguments': {'text': '.'}}])],
        state=STATE, memory=MEMORY, system=SYSTEM,
    )
    return render_prompt(ex)

@torch.no_grad()
def raw_generate(message, temperature=0.0):
    prompt = _build_prompt(message)
    enc = tok(prompt, return_tensors='pt', truncation=True, max_length=4096 - MAX_NEW).to(device)
    kw = dict(max_new_tokens=MAX_NEW, pad_token_id=tok.pad_token_id)
    if temperature and temperature > 0:
        kw.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        kw.update(do_sample=False)
    out = model.generate(**enc, **kw)
    # skip_special_tokens=False: the envelope tags are registered tokens.
    text = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=False)
    for t in (tok.pad_token or '', tok.eos_token or ''):
        if t:
            text = text.replace(t, '')
    end = text.find('</response>')            # the model does not stop; keep the first turn
    return text if end < 0 else text[:end + len('</response>')]

def chat(message, temperature=0.0, show_raw=False):
    text = raw_generate(message, temperature=temperature)
    think = parse_think(text)
    calls = parse_calls(text)
    print('you  >', message)
    if think:
        print('think>', think)
    if calls:
        for c in calls:
            if c.get('name') == 'answer':
                print('bot  >', (c.get('arguments') or {}).get('text', ''))
            else:
                args = ', '.join(f'{k}={v!r}' for k, v in (c.get('arguments') or {}).items())
                print(f'call > {c.get("name")}({args})')
    else:
        print('bot  > (no parseable call)')
    if show_raw:
        print('\n--- raw ---\n' + text)
    return text

## 6. Try it

A few requests to start with. The model is strongest on single, direct tool calls
and weakest at reading `<state>`; both show here.

In [ ]:
chat("What's the weather in Tokyo?")
print()
chat('What is the capital of France, and its population?')
print()
chat('Book me a flight to Berlin.')           # no such tool -- should call answer

## 7. Interactive loop

Run this cell and type at the prompt. Enter `quit` (or an empty line) to stop.
Change `DOMAIN`, `TOOLS`, `STATE`, or `MEMORY` in Section 4 and re-run that cell to
change the world the model sees.

In [ ]:
print('type a message; "quit" or empty to stop\n')
while True:
    try:
        msg = input('you > ').strip()
    except (EOFError, KeyboardInterrupt):
        break
    if not msg or msg.lower() in ('quit', 'exit'):
        break
    chat(msg)
    print()

## 8. Testing grounded calls (optional)

To see whether the model reads live state, give it a `<state>` block and ask a
question whose answer depends on it. Set these in Section 4, or override here and
re-run. This is the behaviour the paper measures, and the one the model is weakest
at, so expect it to often ignore the state.

In [ ]:
# Example: the factory domain with a paused smelter in <state>.
STATE = {'buildings': {'b_004': {'type': 'smelter', 'status': 'paused'}}}
TOOLS = [ANSWER] + DOMAIN_TOOLS['factory']

chat('Is building b_004 currently producing?', show_raw=True)

# reset to the world domain afterwards if you like:
# STATE = None; TOOLS = [ANSWER] + DOMAIN_TOOLS['world']